# Reverse Prompt Designer

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://769audio.vn/upload/Cac_Nguyen_Tc_Khi_Vit_Lnh_Prompt_AI.jpg"> 
</p>
</div>

## Description :

The **Reverse Prompt Designer** is an advanced tool designed to analyze AI-generated text and reconstruct the most likely original prompt that produced it. 

It applies expert-level linguistic forensics and prompt engineering to carefully examine the language, tone, structure, and stylistic features of the provided output.

Key functionalities include:

- **Comprehensive Analysis:** Breaks down AI-generated text to understand its semantics, style, and possible references to cultural or literary sources.

- **Structured Output:** Produces a clear, well-organized Markdown response featuring the original text, best-guess prompt, prompt archetype, prompt variants, detailed reasoning, and notable references.

- **Model Powered:** Utilizes the powerful OpenAI GPT-4o model to deliver accurate and insightful prompt reconstructions.

This app is ideal for prompt engineers, AI developers, and researchers aiming to improve prompt design, gain insights into AI behavior, and enhance the effectiveness of AI-driven content creation.





## Step 1: Environment Setup and Installation

This cell handles initial setup for the notebook:

- Installs dependencies from `requirements/reverse_prompt_designer.requirements.txt`.

- Retries installation up to 3 times on failure.

- Loads environment variables from `.env` using `python-dotenv`.

- Ensures `OPENAI_API_KEY` is set before continuing.

After setup, it clears the output and confirms success.


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
from dotenv import load_dotenv
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]
PROJECT_NAME = "reverse_prompt_designer"
REQUIREMENTS_FILE = f"{PROJECT_NAME}.requirements.txt"


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system(f"pip install -r requirements/{REQUIREMENTS_FILE}")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)


install_requirements()
clear_output()
setup_env()
print("🚀 Setup complete. Continue to the next cell.")

## Step 2: Reverse Prompt Designer – Class Implementation Explained

This step introduces and explains the `ReversePromptDesigner` class, which is responsible for analyzing AI-generated text and reconstructing the original prompt that likely produced it.

- The `__init__` method initializes the class with a default model (`gpt-4o`) and a token limit. It also sets up the OpenAI API client using your environment's API key.

- The `_build_system_message` method defines the AI’s role as a specialist in prompt engineering and linguistic forensics, ensuring focused and expert-level analysis.

- The `_build_user_message` method structures a detailed task prompt that instructs the AI to reverse-engineer a given output. 

  It includes clear Markdown formatting with dedicated sections like Original Output, Best Guess Prompt, Prompt Archetype, Prompt Variants, Reasoning, and References.

- The `generate_reverse_prompt` method sends the structured system and user messages to the OpenAI API and returns a formatted, insightful analysis of the prompt behind the AI-generated text.  Any exceptions are caught and returned as error messages.

This step builds the core intelligence for turning AI outputs into meaningful prompt insights — a valuable tool for refining prompt design workflows.


In [ ]:
from typing import Dict, Union

class ReversePromptDesigner:
    def __init__(self, model: str = "gpt-4o", max_tokens: int = 1000):
        """Initializes the ReversePromptDesigner with the specified model and parameters.
        Args:
            model (str): The OpenAI model to use for generating prompts.
            max_tokens (int): The maximum number of tokens for the generated prompt.
        """
        from openai import OpenAI
        self.model = model
        self.max_tokens = max_tokens
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    
    def _build_system_message(self) -> str:
        return "You are an expert in linguistic forensics and prompt engineering."

    def _build_user_message(self, output_text: str) -> str:
        prompt = f"""You are a world-class AI prompt engineer and linguistic analyst.

    Your task is to reverse-engineer the original prompt that most likely generated the AI-generated output provided below. Carefully analyze the language, semantics, structure, tone, and stylistic choices. Additionally, identify whether the output references any well-known sources such as books, movies, songs, or cultural idioms.
    
    Output Text:
    {output_text.strip()}
    
    Your response must be returned in a clearly structured **Markdown format** with the following sections and spaces for each:
    
    ## 📝 Original Output  
    {output_text.strip()}
    
    ## 🧠 Best Guess Prompt  
    *A reconstructed version of the most likely original prompt.*
    
    ## 🧩 Prompt Archetype  
    *Classify the prompt type — e.g., instructive, poetic, narrative, tweet-style, academic, philosophical, etc.*
    
    ## 🔁 Prompt Variants  
    *Provide three creative, diverse rewordings or alternative versions of the guessed prompt.*
    
    ## 📚 Reasoning  
    *Explain your reasoning for the reconstructed prompt — include analysis of tone, structure, genre, and any contextual clues.*
    
    ## 🧾 References  
    *List any notable references identified in the output (e.g., quotes, idioms, or allusions to books, movies, songs, or cultural elements). If none, state “None detected.”*
    """
        return prompt.strip()

    
    def generate_reverse_prompt(self, input_text: str) -> Union[Dict, Dict[str, str]]:
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": self._build_system_message()},
                    {"role": "user", "content": self._build_user_message(input_text)}
                ],
                max_tokens=self.max_tokens,
                temperature=0.6
            )
            output = response.choices[0].message.content.strip()
            return output
        except Exception as e:
            return {"error": f"An error occurred: {str(e)}"}


## Step 3: Initialize Designer

This step creates an instance of `ReversePromptDesigner` using:

- Uses model `gpt-4o` and `max_tokens=1000`.

- Sets up OpenAI client with your API key.

- Now ready to reverse-engineer any AI-generated output.


In [ ]:
prompt_designer = ReversePromptDesigner()

## Step 4: Run Reverse Prompt Analysis

- The `generate_reverse_prompt` function analyzes the quote.

- It reconstructs the most likely original prompt.

- Displays the result in a well-formatted Markdown
 output with key sections:

  - 📝 Original Output  
  
  - 🧠 Best Guess Prompt  
  
  - 🧩 Prompt Archetype  
  
  - 🔁 Prompt Variants  
  
  - 📚 Reasoning  
  
  - 🧾 References
  


In [ ]:
from IPython.display import display, Markdown

user_input = ""

result = prompt_designer.generate_reverse_prompt(user_input)

display(Markdown(result))

## Conclusion:

The **Reverse Prompt Analyzer** is a powerful tool designed for AI researchers, prompt engineers, and enthusiasts who want to **deconstruct and understand the origins of AI-generated text**. 

With a clean structure and seamless user experience, it offers the following advantages:

### 🔑 Key Highlights:

- **Reverse Engineering Capability**: Converts any AI output into its most likely original prompt using structured linguistic analysis.

- **Markdown-Based Output**: Presents results in a human-readable, sectioned format, aiding readability and clarity.

- **Linguistic Forensics**: Identifies tone, style, archetype, and references hidden within generated content.

- **Creative Prompt Variants**: Suggests diverse alternative prompt formulations for broader ideation and experimentation.

- **Modular & Customizable**: Built with reusable class-based components and environment-driven config, making it easy to extend or integrate into other workflows.

### 💡 Why It’s Useful:

- Helps uncover how a particular AI response was likely triggered.

- Supports educational use in **prompt engineering and model interpretability**.

- Aids in creating safer and more transparent AI systems by analyzing influence and context in outputs.

- Valuable in debugging unexpected completions or content behavior in LLMs.


This app effectively bridges the gap between **AI output and human intent**, providing insight, structure, and inspiration for anyone working with generative AI.


---

# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>